# **Chapter 4 - Building Good Training Datasets – Data Preprocessing**

---

## **Table of Contents**

- [Dealing with missing data](#Dealing-with-missing-data)
  - [Identifying missing values in tabular data](#Identifying-missing-values-in-tabular-data)
  - [Eliminating training examples or features with missing values](#Eliminating-training-examples-or-features-with-missing-values)
  - [Imputing missing values](#Imputing-missing-values)
  - [Understanding the scikit-learn estimator API](#Understanding-the-scikit-learn-estimator-API)
- [Handling categorical data](#Handling-categorical-data)
  - [Nominal and ordinal features](#Nominal-and-ordinal-features)
  - [Mapping ordinal features](#Mapping-ordinal-features)
  - [Encoding class labels](#Encoding-class-labels)
  - [Performing one-hot encoding on nominal features](#Performing-one-hot-encoding-on-nominal-features)
- [Partitioning a dataset into a separate training and test set](#Partitioning-a-dataset-into-seperate-training-and-test-sets)
- [Bringing features onto the same scale](#Bringing-features-onto-the-same-scale)
- [Selecting meaningful features](#Selecting-meaningful-features)
  - [L1 and L2 regularization as penalties against model complexity](#L1-and-L2-regularization-as-penalties-against-model-omplexity)
  - [A geometric interpretation of L2 regularization](#A-geometric-interpretation-of-L2-regularization)
  - [Sparse solutions with L1 regularization](#Sparse-solutions-with-L1-regularization)
  - [Sequential feature selection algorithms](#Sequential-feature-selection-algorithms)
- [Assessing feature importance with Random Forests](#Assessing-feature-importance-with-Random-Forests)
- [Summary](#Summary)

---

The topics that we will cover in this chapter are as follows:

* Removing and imputing missing values from the dataset
* Getting categorical data into shape for machine learning algorithms
* Selecting relevant features for the model construction

In [ ]:
%matplotlib inline

## **Dealing with missing data**

Most computational tools are unable to handle missing values in our datasets or will produce unpredictable results if we simply ignore them. Therefore, it is crucial that we take care of those missing values before we proceed with further analyses. In this section, we will work through several practicals methods to deal with these missing values.

### **Identifying missing values in tabular data**

In [ ]:
import pandas as pd
from io import StringIO
import sys

csv_data = \
'''A,B,C,D
1.0,2.0,3.0,4.0
5.0,6.0,,8.0
10.0,11.0,12.0,'''

# If you are using Python 2.7, you need
# to convert the string to unicode:

if (sys.version_info < (3, 0)):
    csv_data = unicode(csv_data)

df = pd.read_csv(StringIO(csv_data))
df

In [ ]:
df.isnull().sum()
# or
# df.isna().sum()

#### **Convenient data handling with pandas’ DataFrame**

Although scikit-learn was originally developed for working with NumPy arrays only, it can sometimes be more convenient to preprocess data using pandas’ DataFrame. Nowa-
days, most scikit-learn functions support DataFrame objects as inputs, but since NumPy array handling is more mature in the scikit-learn API, it is recommended to use NumPy arrays when possible. Note that you can always access the underlying NumPy array of a DataFrame via the values attribute before you feed it into a scikit-learn estimator:

In [ ]:
# access the underlying NumPy array
# via the `values` attribute
df.values

### **Eliminating training examples or features with missing values**

One of the easiest ways to deal with missing data is simply to remove the corresponding features (columns) or training examples (rows) from the dataset entirely; rows with missing values can easily be dropped via the `dropna` method:

In [ ]:
# remove rows that contain missing values
df.dropna(axis=0)

In [ ]:
# remove columns that contain missing values
df.dropna(axis=1)

In [ ]:
# remove columns that contain missing values
df.dropna(axis=1)

In [ ]:
# only drop rows where all columns are NaN
df.dropna(how='all')

In [ ]:
# drop rows that have fewer than 3 real values 
df.dropna(thresh=4)

In [ ]:
# only drop rows where NaN appear in specific columns (here: 'C')
df.dropna(subset=['C'])

Although the removal of missing data seems to be a convenient approach, it also comes with certain disadvantages; for example, we may end up removing too many samples, which will make a reliable analysis impossible. Or, if we remove too many feature columns, we will run the risk of losing valuable information that our classifier needs to discriminate between classes. In the next section, we will look at one of the most commonly used alternatives for dealing with missing values: interpolation techniques.

### **Imputing missing values**

When dropping data isn't feasible due to information loss, missing value imputation is a viable alternative. A common approach is **mean imputation**, which replaces missing values with the average of that feature column. This can be easily implemented using `scikit-learn`'s `SimpleImputer`:

In [ ]:
# again: our original array
df.values

In [ ]:
# impute missing values via the column mean
from sklearn.impute import SimpleImputer
import numpy as np

imr = SimpleImputer(missing_values=np.nan, strategy='mean')
imr = imr.fit(df.values)
imputed_data = imr.transform(df.values)
imputed_data

Missing values were imputed using column-wise means. The `strategy` parameter also supports `median` or `most_frequent`, with the latter being ideal for categorical features (e.g., color encodings).

Alternatively, we can perform the same mean imputation directly on a pandas DataFrame using the `fillna` method:

In [ ]:
df.fillna(df.mean())

For more advanced techniques, such as the `KNNImputer` (which utilizes a k-nearest neighbors approach), refer to the [scikit-learn imputation documentation](https://scikit-learn.org/stable/modules/impute.html).

### **Understanding the scikit-learn estimator API**

The scikit-learn API provides a consistent framework for data preprocessing and predictive modeling by distinguishing between **transformers** and **estimators**.

#### **1. The Transformer API**
Transformers are classes used for data transformation and preprocessing (e.g., `SimpleImputer` or `StandardScaler`). Their workflow is defined by two primary methods:
* **`fit()`**: This method is used to learn parameters from the training data. For example, a scaler would calculate the sample mean ($\mu$) and standard deviation ($\sigma$) during this stage.
* **`transform()`**: This method applies the learned parameters to transform the data. Crucially, the same parameters obtained from the training set must be reused to transform the test set and any future data to ensure consistency.

<figure style="text-align: center; width: 500px; margin: 0 auto;">
    <img src='figures/04_02.png' style='width:100%'>
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 4.2</strong> Using the scikit-learn API for data transformation.
    </figcaption>
</figure>


#### **2. The Estimator API**
Conceptually similar to transformers, estimators are used for predictive models such as classifiers or regressors.
* **`fit()`**: In supervised learning, this method learns the model parameters by taking both the feature matrix ($X$) and the target labels ($y$) as input.
* **`predict()`**: Once the model is fitted, this method is used to generate predictions (e.g., class labels) for new, unlabeled data instances.

<figure style="text-align: center; width: 500px; margin: 0 auto;">
    <img src='figures/04_03.png' style='width:100%'>
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 4.3</strong> Using the scikit-learn API for predictive models such as classifiers.
    </figcaption>
</figure>


#### **Core Requirements**
A fundamental constraint of the scikit-learn API is that any dataset passed to the `transform()` or `predict()` methods must have the same number of features (dimensions) as the dataset originally used to `fit()` the model. This ensures that the learned parameters map correctly to the input features of new data.

---

## **Handling categorical data**

Real-world datasets frequently incorporate categorical feature columns, which must be distinguished into two primary types: **ordinal** and **nominal**. Ordinal features possess a defined, logical ordering (e.g., T-shirt sizes where $XL > L > M$), whereas nominal features lack any inherent rank or order (e.g., T-shirt colors).

Let's create a DataFrame with ordinal and nominal categorical features. We class labels as last columns:

In [ ]:
import pandas as pd

df = pd.DataFrame([['green', 'M', 10.1, 'class2'],
                   ['red', 'L', 13.5, 'class1'],
                   ['blue', 'XL', 15.3, 'class2']])

df.columns = ['color', 'size', 'price', 'classlabel']
df

### **Mapping ordinal features**

Ordinal features must be transformed from string labels into numerical integers to ensure that learning algorithms correctly interpret their intrinsic rank (e.g., $XL > L > M$). Unlike nominal features, ordinal data requires a mapping function $f: S \to \mathbb{Z}$ where the numerical order reflects the logical hierarchy.

While scikit-learn's `LabelEncoder` or internal estimator logic can handle nominal class labels, manual mapping remains the standard practice for ordinal features to ensure the model captures the quantitative relationships between categories. Therefore, We must define the mapping manually:

In [ ]:
size_mapping = {'XL': 3,
                'L': 2,
                'M': 1}

df['size'] = df['size'].map(size_mapping)
df

To restore the original string representation for model interpretation, apply an inverse mapping dictionary using a dictionary comprehension:


In [ ]:
inv_size_mapping = {v: k for k, v in size_mapping.items()}
df['size'].map(inv_size_mapping)

### **Encoding class labels**

Class labels must be encoded as integers to satisfy the requirements of most machine learning libraries and improve computational performance. Because class labels are nominal features, the mapping $f: S \to \mathbb{Z}$ can assign integers arbitrarily without preserving a specific order.

#### **Manual Mapping with pandas**
1.  **Define Mapping:** Construct a dictionary by enumerating unique labels.
2.  **Transform and Reverse:** Use the `pandas` `.map()` method to apply the integer transformation to the target column. To restore the original labels, apply an inverse mapping dictionary:

In [ ]:
import numpy as np

# create a mapping dict
# to convert class labels from strings to integers
class_mapping = {label: idx for idx, label in enumerate(np.unique(df['classlabel']))}
class_mapping

In [ ]:
# to convert class labels from strings to integers
df['classlabel'] = df['classlabel'].map(class_mapping)
df

In [ ]:
# reverse the class label mapping
inv_class_mapping = {v: k for k, v in class_mapping.items()}
df['classlabel'] = df['classlabel'].map(inv_class_mapping)
df

#### **Automated Mapping with scikit-learn**
The `LabelEncoder` class in the `preprocessing` module provides a more streamlined transformer for this task.
*   **`fit_transform()`**: This method learns the unique classes from the input array and immediately transforms them into an integer array.
*   **`inverse_transform()`**: This method is used to project the integer indices back onto the original categorical string labels for model evaluation or interpretation.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Label encoding with sklearn's LabelEncoder
class_le = LabelEncoder()
y = class_le.fit_transform(df['classlabel'].values)
y

In [ ]:
# reverse mapping
class_le.inverse_transform(y)

### **Performing one-hot encoding on nominal features**

Encoding nominal features as ordered integers (e.g., $blue=0, green=1, red=2$) is a common error, as learning algorithms will assume a false ordinal relationship ($2 > 1 > 0$), leading to suboptimal performance. One-hot encoding resolves this by creating a binary "dummy" feature for each unique category in the column.

In [ ]:
X = df[['color', 'size', 'price']].values
color_le = LabelEncoder()
X[:, 0] = color_le.fit_transform(X[:, 0])
X

scikit-learn `OneHotEncoder` is used to transform nominal features into a sparse or dense array of binary indicators. To selectively transform specific columns while leaving others (like numerical features) untouched, use the `ColumnTransformer` class with the `'passthrough'` argument.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

X = df[['color', 'size', 'price']].values
color_ohe = OneHotEncoder()
color_ohe.fit_transform(X[:, 0].reshape(-1, 1)).toarray()

In [ ]:
from sklearn.compose import ColumnTransformer

X = df[['color', 'size', 'price']].values
c_transf = ColumnTransformer([ ('onehot', OneHotEncoder(), [0]),
                               ('nothing', 'passthrough', [1, 2])])
c_transf.fit_transform(X).astype(float)

A convenient alternative is pandas `get_dummies` that automatically converts string columns into dummy features while preserving numerical columns.

In [ ]:
pd.get_dummies(df[['price', 'color', 'size']])

One-hot encoding introduces multicollinearity, where features are highly correlated. This can cause numerical instability in algorithms that require matrix inversion. To mitigate this, one redundant column should be dropped (reducing $K$ categories to $K-1$ features) without losing information.

*   **In pandas:** we pass `drop_first=True` to the `get_dummies` function.
*   **In scikit-learn:** We initialize `OneHotEncoder(drop='first', categories='auto')`.

In [ ]:
# multicollinearity guard in get_dummies

pd.get_dummies(df[['price', 'color', 'size']], drop_first=True)

In [ ]:
# multicollinearity guard for the OneHotEncoder

color_ohe = OneHotEncoder(categories='auto', drop='first')
c_transf = ColumnTransformer([ ('onehot', color_ohe, [0]),
                               ('nothing', 'passthrough', [1, 2])])
c_transf.fit_transform(X).astype(float)

>**Note:** For high-cardinality nominal features where one-hot encoding leads to an excessive number of dimensions, alternative encoding schemes provide a more compact representation and serve as additional hyperparameters for performance tuning. Alternative Encoding Schemes:
>
>*   **Binary Encoding**: This method converts integer-mapped categories into binary representations, where each bit position forms a new feature column. It significantly reduces the dimensionality of the feature space from $K - 1$ to approximately $\log_2(K)$ columns, where $K$ is the number of unique categories.
>*   **Count (Frequency) Encoding**: This technique replaces each categorical label with its absolute frequency (or count) within the training set.
>
>
>While scikit-learn's `preprocessing` module focuses on `OneHotEncoder` and `LabelEncoder`, these advanced schemes are available through the scikit-learn-compatible `category_encoders` library. 
>
>Because no single encoding method is guaranteed to yield superior results, the choice of scheme should be treated as a hyperparameter to be optimized during the model selection process.

### **Optional: Encoding Ordinal Features**

If we are unsure about the numerical differences between the categories of ordinal features, or the difference between two ordinal values is not defined, we can also encode them using a threshold encoding with 0/1 values. For example, we can split the feature "size" with values M, L, and XL into two new features "x > M" and "x > L". Let's consider the original DataFrame:

In [ ]:
df = pd.DataFrame([['green', 'M', 10.1, 'class2'],
                   ['red', 'L', 13.5, 'class1'],
                   ['blue', 'XL', 15.3, 'class2']])

df.columns = ['color', 'size', 'price', 'classlabel']
df

We can use the `apply` method of pandas' DataFrames to write custom lambda expressions in order to encode these variables using the value-threshold approach:

In [ ]:
df['x > M'] = df['size'].apply(lambda x: 1 if x in {'L', 'XL'} else 0)
df['x > L'] = df['size'].apply(lambda x: 1 if x == 'XL' else 0)

del df['size']
df

---

## **Partitioning a dataset into a separate training and test set**

To ensure a model generalizes to unseen data and to identify overfitting (high variance), datasets must be partitioned into disjoint subsets for training and evaluation.

In this section, we will prepare a new dataset, the **Wine** dataset. The Wine dataset is another open-source dataset that is available from the UCI machine learning repository (https://archive.ics.uci.edu/ml/datasets/Wine); it consists of 178 wine examples with 13 features describing their different chemical properties. The following cell downloads and reads in raw data using directly from the url. Raw data is also available locally in case the url breaks or data is not available online.

In [ ]:
import pandas as pd

# The following method is restricted by host proxy.
# df = pd.read_csv(
#     'https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data',
#     header=None
# )

# Read local raw data
df_wine = pd.read_csv('data/wine.data')
df_wine.columns = [
    'Class label', 'Alcohol',
    'Malic acid', 'Ash',
    'Alcalinity of ash', 'Magnesium',
    'Total phenols', 'Flavanoids',
    'Nonflavanoid phenols',
    'Proanthocyanins',
    'Color intensity',
    'Hue',
    'OD280/OD315 of diluted wines',
    'Proline'
    ]
print("class labels", df_wine['Class label'].unique())
df_wine.head()

In [ ]:
print(df_wine.info())

In [ ]:
print(df_wine['Class label'].value_counts(normalize=True))


The objective is to estimate the generalization error by withholding a portion of the data during the model-fitting stage. 

*   **Training Set**: Used to learn model parameters.
*   **Test Set**: Serves as an "unseen" proxy for the real world to provide an unbiased evaluation of the final model.

In scikit-learn, this is implemented via the `sklearn.model_selection` module. 

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df_wine.iloc[:, 1:].values, df_wine.iloc[:, 0].values

X_train, X_test, y_train, y_test =train_test_split(
    X, y, 
    test_size=0.3, 
    random_state=0, 
    stratify=y)

print(X_train .shape)
print(X_test.shape)

**Key Hyperparameters:**
*   **`test_size`**: The proportion of the dataset allocated to the test split (e.g., $0.3$ for a 70:30 split).
*   **`random_state`**: A fixed seed for the internal pseudo-random number generator to ensure reproducibility.
*   **`stratify`**: Ensures the training and test subsets maintain the same class proportions as the original dataset (i.e., preserving the distribution $P(y)$), which is critical for imbalanced datasets.

The choice of split ratio is a trade-off between providing enough data for the algorithm to learn and enough data to achieve a reliable performance estimate.

*   **Standard Datasets**: Common ratios include 60:40, 70:30, or 80:20.
*   **Large Datasets ($N > 100,000$)**: Ratios such as 90:10 or 99:1 are appropriate, as even 1% of a massive dataset can yield a statistically significant estimate of generalization performance.

### **Model Refitting**

A standard practice after estimating the generalization error is to retrain the classifier on the **entire dataset** (training + test). This maximizes the information available to the learning algorithm, though it leaves no remaining independent data for a final "unbiased" evaluation of that specific refitted instance.

---

## **Bringing features onto the same scale**

Feature scaling is essential for optimal performance in most machine learning and optimization algorithms, as features with larger magnitudes can otherwise dominate loss functions (e.g., in gradient descent) or distance metrics (e.g., in KNN). Notable exceptions are decision trees and random forests, which are scale-invariant.


### **Normalization (Min-Max Scaling)**

Normalization rescales features to a fixed bounded interval, typically $[0, 1]$. It is mathematically defined as:

$$x_{norm}^{(i)} = \frac{x^{(i)} - x_{min}}{x_{max} - x_{min}}$$
*   **scikit-learn API**: `sklearn.preprocessing.MinMaxScaler`.

### **Standardization**

Standardization centers features at a mean ($\mu$) of 0 with a standard deviation ($\sigma$) of 1, resulting in unit variance. The transformation is defined as:

$$x_{std}^{(i)} = \frac{x^{(i)} - \mu_x}{\sigma_x}$$

Standardization is often more practical for optimization algorithms like gradient descent because many linear models initialize weights to 0 or small random values near 0. Unlike min-max scaling, standardization maintains information about outliers.
*   **scikit-learn API**: `sklearn.preprocessing.StandardScaler`.

### **Robust Scaling**

For small datasets containing many outliers, `RobustScaler` is recommended. It independently removes the median from each feature and scales the data according to the Interquartile Range (IQR), specifically the 1st quartile (25th quantile) and 3rd quartile (75th quantile), making extreme values less pronounced.
*   **scikit-learn API**: `sklearn.preprocessing.RobustScaler`.

### **Procedural Consistency**

A critical requirement in scikit-learn's workflow is that scaling parameters (such as $\mu$ and $\sigma$) must be estimated strictly from the **training dataset** using the `fit()` method. These same parameters must then be reused via the `transform()` method for the test dataset and any new data instances to ensure the values remain comparable.

In [ ]:
# Prepare data if not in globals namespace
if 'df_wine' not in globals():
    
    # Read in data
    df_wine = pd.read_csv('data/wine.data')
    df_wine.columns = [
        'Class label', 'Alcohol',
        'Malic acid', 'Ash',
        'Alcalinity of ash', 'Magnesium',
        'Total phenols', 'Flavanoids',
        'Nonflavanoid phenols',
        'Proanthocyanins',
        'Color intensity',
        'Hue',
        'OD280/OD315 of diluted wines',
        'Proline'
    ]

    # Define features and label sets
    X = df_wine.drop(columns=['Class label'])
    y = df_wine['Class label']
    assert X.shape == (177, 13)
    assert y.shape == (177,)

    # Split data into train/test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

* **Min-Max Scaling**

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

mms = MinMaxScaler()
X_train_norm = mms.fit_transform(X_train)
X_test_norm = mms.transform(X_test)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4), squeeze=True)
ax1.hist(X_train.to_numpy()[:, 0], bins=20);
ax1.set_xlabel('Alcohol')
ax2.hist(X_train_norm[:, 0], bins=20);
ax2.set_xlabel('Normalized Alcohol');

* **Standard Scaling**

In [ ]:
from sklearn.preprocessing import StandardScaler

stdsc = StandardScaler()
X_train_std = stdsc.fit_transform(X_train)
X_test_std = stdsc.transform(X_test)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4), squeeze=True)
ax1.hist(X_train.to_numpy()[:, 0], bins=20);
ax1.set_xlabel('Alcohol')
ax2.hist(X_train_std[:, 0], bins=20);
ax2.set_xlabel('Standardized Alcohol');

A visual example:

In [ ]:
import numpy as np


ex = np.array([0, 1, 2, 3, 4, 5])

print('standardized:', (ex - ex.mean()) / ex.std())

# Please note that pandas uses ddof=1 (sample standard deviation) 
# by default, whereas NumPy's std method and the StandardScaler
# uses ddof=0 (population standard deviation)

# normalize
print('normalized:', (ex - ex.min()) / (ex.max() - ex.min()))

---

# Selecting meaningful features

...

## L1 and L2 regularization as penalties against model complexity

## A geometric interpretation of L2 regularization

In [ ]:
Image(filename='figures/04_05.png', width=500) 

In [ ]:
Image(filename='figures/04_06.png', width=500) 

## Sparse solutions with L1-regularization

In [ ]:
Image(filename='figures/04_06.png', width=500) 

For regularized models in scikit-learn that support L1 regularization, we can simply set the `penalty` parameter to `'l1'` to obtain a sparse solution:

In [ ]:
from sklearn.linear_model import LogisticRegression

LogisticRegression(penalty='l1')

Applied to the standardized Wine data ...

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(penalty='l1', C=1.0, solver='liblinear', multi_class='ovr')
# Note that C=1.0 is the default. You can increase
# or decrease it to make the regulariztion effect
# weaker or stronger, respectively.
lr.fit(X_train_std, y_train)
print('Training accuracy:', lr.score(X_train_std, y_train))
print('Test accuracy:', lr.score(X_test_std, y_test))

In [ ]:
lr.intercept_

In [ ]:
np.set_printoptions(8)

In [ ]:
lr.coef_[lr.coef_!=0].shape

In [ ]:
lr.coef_

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure()
ax = plt.subplot(111)
    
colors = ['blue', 'green', 'red', 'cyan', 
          'magenta', 'yellow', 'black', 
          'pink', 'lightgreen', 'lightblue', 
          'gray', 'indigo', 'orange']

weights, params = [], []
for c in np.arange(-4., 6.):
    lr = LogisticRegression(penalty='l1', C=10.**c, solver='liblinear', 
                            multi_class='ovr', random_state=0)
    lr.fit(X_train_std, y_train)
    weights.append(lr.coef_[1])
    params.append(10**c)

weights = np.array(weights)

for column, color in zip(range(weights.shape[1]), colors):
    plt.plot(params, weights[:, column],
             label=df_wine.columns[column + 1],
             color=color)
plt.axhline(0, color='black', linestyle='--', linewidth=3)
plt.xlim([10**(-5), 10**5])
plt.ylabel('Weight coefficient')
plt.xlabel('C (inverse regularization strength)')
plt.xscale('log')
plt.legend(loc='upper left')
ax.legend(loc='upper center', 
          bbox_to_anchor=(1.38, 1.03),
          ncol=1, fancybox=True)

#plt.savefig('figures/04_08.png', dpi=300, 
#            bbox_inches='tight', pad_inches=0.2)

plt.show()

<br>
<br>

## Sequential feature selection algorithms

In [ ]:
from sklearn.base import clone
from itertools import combinations
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


class SBS:
    def __init__(self, estimator, k_features, scoring=accuracy_score,
                 test_size=0.25, random_state=1):
        self.scoring = scoring
        self.estimator = clone(estimator)
        self.k_features = k_features
        self.test_size = test_size
        self.random_state = random_state

    def fit(self, X, y):
        
        X_train, X_test, y_train, y_test = \
            train_test_split(X, y, test_size=self.test_size,
                             random_state=self.random_state)

        dim = X_train.shape[1]
        self.indices_ = tuple(range(dim))
        self.subsets_ = [self.indices_]
        score = self._calc_score(X_train, y_train, 
                                 X_test, y_test, self.indices_)
        self.scores_ = [score]

        while dim > self.k_features:
            scores = []
            subsets = []

            for p in combinations(self.indices_, r=dim - 1):
                score = self._calc_score(X_train, y_train, 
                                         X_test, y_test, p)
                scores.append(score)
                subsets.append(p)

            best = np.argmax(scores)
            self.indices_ = subsets[best]
            self.subsets_.append(self.indices_)
            dim -= 1

            self.scores_.append(scores[best])
        self.k_score_ = self.scores_[-1]

        return self

    def transform(self, X):
        return X[:, self.indices_]

    def _calc_score(self, X_train, y_train, X_test, y_test, indices):
        self.estimator.fit(X_train[:, indices], y_train)
        y_pred = self.estimator.predict(X_test[:, indices])
        score = self.scoring(y_test, y_pred)
        return score

In [ ]:
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)

# selecting features
sbs = SBS(knn, k_features=1)
sbs.fit(X_train_std, y_train)

# plotting performance of feature subsets
k_feat = [len(k) for k in sbs.subsets_]

plt.plot(k_feat, sbs.scores_, marker='o')
plt.ylim([0.7, 1.02])
plt.ylabel('Accuracy')
plt.xlabel('Number of features')
plt.grid()
plt.tight_layout()
# plt.savefig('figures/04_09.png', dpi=300)
plt.show()

In [ ]:
k3 = list(sbs.subsets_[10])
print(df_wine.columns[1:][k3])

In [ ]:
knn.fit(X_train_std, y_train)
print('Training accuracy:', knn.score(X_train_std, y_train))
print('Test accuracy:', knn.score(X_test_std, y_test))

In [ ]:
knn.fit(X_train_std[:, k3], y_train)
print('Training accuracy:', knn.score(X_train_std[:, k3], y_train))
print('Test accuracy:', knn.score(X_test_std[:, k3], y_test))

<br>
<br>

# Assessing feature importance with Random Forests

In [ ]:
from sklearn.ensemble import RandomForestClassifier

feat_labels = df_wine.columns[1:]

forest = RandomForestClassifier(n_estimators=500,
                                random_state=1)

forest.fit(X_train, y_train)
importances = forest.feature_importances_

indices = np.argsort(importances)[::-1]

for f in range(X_train.shape[1]):
    print("%2d) %-*s %f" % (f + 1, 30, 
                            feat_labels[indices[f]], 
                            importances[indices[f]]))

plt.title('Feature importance')
plt.bar(range(X_train.shape[1]), 
        importances[indices],
        align='center')

plt.xticks(range(X_train.shape[1]), 
           feat_labels[indices], rotation=90)
plt.xlim([-1, X_train.shape[1]])
plt.tight_layout()
# plt.savefig('figures/04_10.png', dpi=300)
plt.show()

In [ ]:
from sklearn.feature_selection import SelectFromModel

sfm = SelectFromModel(forest, threshold=0.1, prefit=True)
X_selected = sfm.transform(X_train)
print('Number of features that meet this threshold criterion:', 
      X_selected.shape[1])

Now, let's print the 3 features that met the threshold criterion for feature selection that we set earlier (note that this code snippet does not appear in the actual book but was added to this notebook later for illustrative purposes):

In [ ]:
for f in range(X_selected.shape[1]):
    print("%2d) %-*s %f" % (f + 1, 30, 
                            feat_labels[indices[f]], 
                            importances[indices[f]]))

<br>
<br>

# Summary

...

---

Readers may ignore the next cell.

In [ ]:
! python ../.convert_notebook_to_script.py --input ch04.ipynb --output ch04.py